# 03 · Reversible transformer at its maximum batch size

1. **Memory vs depth**: peak memory at a small fixed batch for L = 4, 8, 16, 32, Euler vs the chosen reversible variant. This checks the lesson's claim that depth drops out of the activation memory.
2. **Max batch**: search for the largest batch that fits and train on the same 50M tokens.

The token budget is fixed, so a larger batch means **fewer optimizer steps**. At roughly 1000 sequences per batch, 50M tokens is only about 100 steps. The LR is scaled by √(B/B_fixed) (capped at 3e-3), with 10% warm-up. Expect the final loss to be worse than run 2's: that is the trade-off being measured. Notebook 04 plots loss against both tokens and optimizer steps.

In [ ]:
# --- Colab setup: GPU runtime (Runtime > Change runtime type > T4 GPU) ---
REPO_URL = "https://github.com/gaurkhare/gaurav-eagv5-s13.git"
import os, sys, json
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/era_a13"          # data + results persist across notebooks
    if not os.path.exists("/content/repo"):
        !git clone -q {REPO_URL} /content/repo
    os.chdir("/content/repo")
else:
    WORK = os.path.abspath("..")                      # running locally from notebooks/
    os.chdir(WORK)
sys.path.insert(0, os.getcwd())
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
DATA_DIR, RESULTS = f"{WORK}/data", f"{WORK}/results"
os.makedirs(RESULTS, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no nvidia GPU"

In [ ]:
import math, torch
from revllm import ModelConfig, TrainConfig, train, find_max_batch, fits
B_FIXED = json.load(open(f"{RESULTS}/batch.json"))["fixed_batch"]
vj = json.load(open(f"{RESULTS}/variant.json"))
BEST, KW = vj["best_variant"], vj["model_kwargs"]
print("variant:", BEST, KW, "| fixed batch:", B_FIXED)

## Peak memory vs depth

In [ ]:
B_DEPTH = 16
depth = {"euler": {}, BEST: {}}
for L in [4, 8, 16, 32]:
    for name, kw in [("euler", dict(residual="euler")), (BEST, KW)]:
        ok, peak = fits(ModelConfig(n_layers=L, **kw), B_DEPTH)
        depth[name][L] = peak
        print(f"L={L:2d} {name:22s} {'%.2f GB' % peak if ok else 'OOM'}")
json.dump({"batch": B_DEPTH, **depth}, open(f"{RESULTS}/mem_vs_depth.json", "w"), indent=1)

## Max batch search and training

In [ ]:
mcfg = ModelConfig(**KW)
bmax, trials = find_max_batch(mcfg, start=B_FIXED)
print(f"reversible ({BEST}) max batch = {bmax}  ({bmax/B_FIXED:.1f}x the fixed batch)")
bj = json.load(open(f"{RESULTS}/batch.json")); bj.update(rev_max_batch=bmax, rev_trials=trials)
json.dump(bj, open(f"{RESULTS}/batch.json", "w"), indent=1)

In [ ]:
def run(B):
    lr = min(3e-3, 1e-3 * math.sqrt(B / B_FIXED))
    steps = 50_000_000 // (B * mcfg.seq_len)
    print(f"B={B}, lr={lr:.2e}, optimizer steps={steps} (run 2 had {50_000_000 // (B_FIXED * mcfg.seq_len)})")
    cfg = TrainConfig(name="reversible_maxbatch", data_dir=DATA_DIR, out_dir=RESULTS, batch_size=B,
                      total_tokens=50_000_000, lr=lr, warmup_frac=0.1, eval_every=max(5, steps // 20),
                      log_every=max(1, min(20, steps // 100)), model=mcfg)
    return train(cfg)

# find_max_batch ran two full train steps at bmax, so the run should fit; the fallback only guards against
# allocator fragmentation late in a long run. Whatever batch actually trains is recorded next to bmax.
B_MAX = bmax
try:
    res, model = run(B_MAX)
except torch.OutOfMemoryError:
    torch.cuda.empty_cache()
    B_MAX = int(0.9 * bmax) // 8 * 8
    print(f"OOM at the searched max {bmax}; retrying at {B_MAX}")
    res, model = run(B_MAX)
bj = json.load(open(f"{RESULTS}/batch.json")); bj.update(rev_train_batch=B_MAX)
json.dump(bj, open(f"{RESULTS}/batch.json", "w"), indent=1)